In [1]:
from configuracoes_notebooks import set_proj_dir
set_proj_dir()

O diretorio do seu projeto é coleta_cebrap
Caminho absoluto do diretorio encontrado C:\Users\x526378\Desktop\projetos\cebrap\coleta_cebrap
Caminho no path.


In [2]:
import geopandas as gpd
import pandas as pd

from notebooks.jupyter import utils
from utils import (
    get_data_diretorio,
    check_crs,
    save_parquet_excel,
)
from utils.downloads import download_malha_geosampa

Eu ainda não tenho certeza de quais camadas devo pegar, vou pegar todas e comparar

In [3]:
data_path = get_data_diretorio()

# Risco de Ocorrência de Inundação

In [4]:
gdf_risco_inund= download_malha_geosampa(
    'risco_ocorrencia_inundacao', 
    data_path
)

helloo
C:\Users\x526378\Desktop\projetos\cebrap\coleta_cebrap\data\cache\risco_ocorrencia_inundacao.zip


In [5]:
gdf_risco_inund.shape

(308, 8)

In [6]:
gdf_risco_inund.sample(5)

,cd_identif,dt_ocorren,dc_tipo_oc,nm_distrit,dt_carga,nm_subpref,sg_fonte_o,geometry
259,594,2025-01-24,INUNDACAO,None,2025-03-14,AF - ARICANDUVA/VILA FORMOSA,SIGRC,POINT (341502.547 7396287.733)
162,1454,2025-02-07,INUNDACAO,None,2025-03-14,MP - SAO MIGUEL PAULISTA,SIGRC,POINT (351432.097 7399392.634)
264,633,2025-01-25,INUNDACAO,None,2025-03-14,MG - VILA MARIA/VILA GUILHERME,SIGRC,POINT (340593.119 7399517.273)
20,884,2025-02-01,INUNDACAO,None,2025-03-14,MP - SAO MIGUEL PAULISTA,SIGRC,POINT (351829.937 7401680.191)
171,1513,2025-02-10,INUNDACAO,None,2025-03-14,CT - CIDADE TIRADENTES,SIGRC,POINT (358046.669 7390558.67)


# Conferir dados

Vamos filtrar pelas datas para ver qual a data de ocorrência mais recente e a mais antiga.

In [7]:
min_data=gdf_risco_inund['dt_ocorren'].min()
gdf_risco_inund.loc[gdf_risco_inund['dt_ocorren']==min_data]

,cd_identif,dt_ocorren,dc_tipo_oc,nm_distrit,dt_carga,nm_subpref,sg_fonte_o,geometry
214,16,2025-01-02,INUNDACAO,None,2025-03-14,IQ - ITAQUERA,SIGRC,POINT (350554.77 7397886.967)


In [8]:
max_data=gdf_risco_inund['dt_ocorren'].max()
gdf_risco_inund.loc[gdf_risco_inund['dt_ocorren']==max_data]

,cd_identif,dt_ocorren,dc_tipo_oc,nm_distrit,dt_carga,nm_subpref,sg_fonte_o,geometry
212,2097,2025-02-27,INUNDACAO,None,2025-03-14,CV - CASA VERDE/CACHOEIRINHA,SIGRC,POINT (329182.739 7400465.875)
213,2102,2025-02-27,INUNDACAO,None,2025-03-14,PI - PINHEIROS,SIGRC,POINT (325067.722 7395845.033)


In [9]:
print(
    f'''
A ocorrência mais antiga é de: {min_data}
A ocorrência mais recente é de: {max_data}
    '''
)


A ocorrência mais antiga é de: 2025-01-02 00:00:00
A ocorrência mais recente é de: 2025-02-27 00:00:00
    


Como podemos ver, estes dados são referentes às ocorrências de inundação entre os janeiro e fevereiro de 2025.

# Padronização de nomes

Iremos apagar as colunas de tipo de ocorrência (todas as ocorrências são Inundações); nome dos distrito (não consta); data de carga; e fonte de origem dos dados (todos da mesma fonte)

In [10]:
drop_cols={
    'dc_tipo_oc',
    'nm_distrit',
    'dt_carga',
    'sg_fonte_o'
}

gdf_risco_inund.rename(
    {'cd_identif':'cd_inund_risco'},
    axis=1,
    inplace=True
)
gdf_risco_inund.drop(columns=drop_cols, axis=1, inplace=True)

In [11]:
gdf_risco_inund.sample(2)

,cd_inund_risco,dt_ocorren,nm_subpref,geometry
152,1384,2025-02-06,BT - BUTANTA,POINT (321835.648 7391954.441)
134,1245,2025-02-03,MP - SAO MIGUEL PAULISTA,POINT (357241.741 7401532.248)


# Conferir CRS

In [12]:
gdf_risco_inund=check_crs(gdf_risco_inund)

# Salvar GDF

In [13]:
save_parquet_excel(
    gdf_risco_inund,
    'pontos_risco_inundacao',
    data_path,
    data_subpath='assets'
)